# Data for the paper figures

The pipeline saves its fits but not three things the paper-style figures need. This notebook computes them from the final point and stores them in `OUT/paper_figs_data/` (port of `v3_paper_data.py`):

1. **Relative reduced sensitivities** $\partial\ln y^*/\partial\ln\theta$ of every measured output at the final point, per condition, for the free parameters (Fig. 2a). Computed from the profiled (reduced) model of `sensitivity.py`, the same object the Fisher information is built from, but *relative* (divided by the output). Relative sensitivities diverge as an output approaches zero, which is why the figure excludes entries below 1% of the output median.
2. **The literature-parameter model.** All 37 parameters fixed at the literature values (`Data/theta_init_sampled.csv`, clipped into the theta bounds), the inner problem over $x, u, b$ solved under the same bounds: its predicted outputs (`y_literature_inner.csv`) and its objective breakdown (the "fixed estimation" of the paper, here with the repaired bounds).
3. **Metrics per stage** (literature, stage 0, final, LOOCV and the LOOCV training-mean baseline) on identical definitions: RMSE and $R^2$ against the per-output mean of the in-sample data (`stage_metrics.csv`).

Parts 1 and 2 solve inner problems and run behind `HK_RUN=1` (a few minutes); part 3 is arithmetic on saved files and is recomputed every time, so **re-run the notebook (no `HK_RUN` needed) after `loocv.ipynb`** to refresh `stage_metrics.csv`.

In [1]:
import os
import sys
import logging
import warnings

HERE = os.getcwd()
REPO = os.path.dirname(HERE) if os.path.basename(HERE) == "notebooks" else HERE
for path in (os.path.join(REPO, "notebooks"), os.path.join(REPO, "src")):
    if path not in sys.path:
        sys.path.insert(0, path)

import settings  # noqa: F401  (pins BLAS / OpenMP to one thread before numpy is loaded)
from paths import check_out_dir, resolve_out

OUT = resolve_out(os.environ.get("HK_OUT") or settings.RUN["out_dir"])   # default: out_dir of settings/run.csv; relative paths are relative to the repo root
RUN = os.environ.get("HK_RUN", "0") == "1"

try:
    check_out_dir(OUT)
    WRITABLE = True
except ValueError:
    WRITABLE = False

if RUN and not WRITABLE:
    raise ValueError("HK_RUN=1 refuses an output directory inside results/, results_v2/ or results_v3/: %s" % OUT)

warnings.filterwarnings("ignore")
logging.getLogger("pyomo").setLevel(logging.ERROR)
print("REPO     :", REPO)
print("OUT      :", OUT)
print("RUN      :", RUN, "(False = load the saved results of OUT; True = compute and save into OUT)")
print("WRITABLE :", WRITABLE, "(tables / figures / report are written to OUT only if True)")

import json

import numpy as np
import pandas as pd

from settings import THETA_BOUNDS, RUN as RUN_SETTINGS, default_params
from model.parameters import ALL_PARAMS
from estimation.anchors import load_point, read_anchor, check_bounds
from sensitivity import reduced_sensitivity
from metrics import pred_metrics

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

FINAL = os.path.join(OUT, "final")
DATA_OUT = os.path.join(OUT, "paper_figs_data")
N_JOBS = int(os.environ.get("HK_N_JOBS", RUN_SETTINGS["n_jobs"]))

def to_wide(y_by_condition):
    """{condition: Series(output -> value)} -> DataFrame (conditions x outputs)."""
    return pd.DataFrame(y_by_condition).T

def mean_abs_ignoring_nan(values):
    return float(np.nanmean(np.abs(values)))

REPO     : /Users/gabbi/Desktop/Repos/HybridKinetics-IIQ3733
OUT      : /Users/gabbi/Desktop/Repos/HybridKinetics-IIQ3733/results_v4/flux_model
RUN      : True (False = load the saved results of OUT; True = compute and save into OUT)
WRITABLE : True (tables / figures / report are written to OUT only if True)


## 1. Sensitivities and the literature model (`HK_RUN=1`)

`load_point` rebuilds the final point as an estimator without a solve; `reduced_sensitivity(..., relative=True)` re-solves the inner problem of each condition at perturbed $\ln\theta$ (parallel, `params["n_jobs"]`; the step and the consistency settings are those of the selection, `default_params("sensitivity")`), so the table has one row per (condition, output) and one column per free parameter. For the literature model only the objective breakdown and the unperturbed outputs `y0` are needed (one column, `kcat_f_4`, is requested to trigger the solves at the literature theta).

In [2]:
# Settings of the reduced sensitivities (defaults from settings/thresholds.csv)
params = default_params("sensitivity")
params["sens_step"] = 0.01   # finite-difference step in ln theta
params["n_jobs"] = N_JOBS    # worker processes (HK_N_JOBS)
params

{'n_jobs': 8,
 'sens_step': 0.01,
 'sens_step_check': 0.003,
 'fd_consistency_rtol': 0.1,
 'fd_consistency_atol': 0.001}

In [3]:
if RUN:
    check_bounds()
    os.makedirs(DATA_OUT, exist_ok=True)
    state = read_anchor(FINAL)
    est = load_point(state)
    print("final point loaded: obj %.5f, %d free" % (state["obj"], len(state["free"])))

    # ---- relative reduced sensitivities at the final point ----
    final_sensitivity = reduced_sensitivity(est, params, relative=True, check=False)
    blocks = []
    for condition, table in final_sensitivity["sensitivities"].items():
        block = table.copy()
        block["condition"] = condition
        block["output"] = block.index
        blocks.append(block)
    long = pd.concat(blocks, ignore_index=True)
    long.to_csv(os.path.join(DATA_OUT, "sens_rel_by_condition.csv"), index=False)
    mean_abs = long.drop(columns="condition").groupby("output").agg(mean_abs_ignoring_nan)
    mean_abs.to_csv(os.path.join(DATA_OUT, "sens_rel_mean_abs.csv"))
    to_wide(final_sensitivity["y0"]).to_csv(os.path.join(DATA_OUT, "y_final_inner.csv"))
    print("sensitivities: %d conditions, outputs %s" % (len(final_sensitivity["sensitivities"]), list(mean_abs.index)))

    # ---- the literature-parameter model ----
    literature = pd.read_csv(os.path.join(REPO, "Data", "theta_init_sampled.csv"), index_col=0)["value"]
    theta_lit = {}
    for p in ALL_PARAMS:
        theta_lit[p] = float(min(max(float(literature[p]), THETA_BOUNDS[p][0]), THETA_BOUNDS[p][1]))
    literature_model = reduced_sensitivity(est, params, theta=theta_lit, param_names=["kcat_f_4"], relative=True,
                                           check=False)
    to_wide(literature_model["y0"]).to_csv(os.path.join(DATA_OUT, "y_literature_inner.csv"))
    report = literature_model["report"]
    print("literature model: n_nan_entries %s, n_failed_first_pass %s" % (report["n_nan_entries"],
                                                                        report["n_failed_first_pass"]))
    est.objective_breakdown(theta=theta_lit).to_csv(os.path.join(DATA_OUT, "objective_breakdown_literature.csv"))
else:
    print("RUN is False: sensitivities and the literature model are loaded from", DATA_OUT)

[     1s] check_bounds_contain_data: OK (balanced, cofactors, drains); Pi bounds (1.0, 20.0)


final point loaded: obj 10.94288, 11 free


sensitivities: 22 conditions, outputs ['C_3pg', 'C_dhap', 'C_f6p', 'C_fbp', 'C_g6p', 'C_pep', 'v_gapA', 'v_gpmA', 'v_pfkB', 'v_pgi', 'v_pts']


literature model: n_nan_entries 0, n_failed_first_pass 0


### Saved sensitivities and the literature model

In [4]:
def read_saved(name, **kw):
    path = os.path.join(DATA_OUT, name)
    if os.path.exists(path):
        return pd.read_csv(path, **kw)
    print("not found (computed with HK_RUN=1): paper_figs_data/" + name)
    return None

sens_mean = read_saved("sens_rel_mean_abs.csv", index_col=0)
if sens_mean is not None:
    print("mean |d ln y / d ln theta| over the conditions (rows: outputs, columns: free parameters)")
    display(sens_mean)

bd_lit = read_saved("objective_breakdown_literature.csv", index_col=0)
if bd_lit is not None:
    print("objective of the literature model (mean over conditions): %.2f" % bd_lit.loc["MEAN", "total"])

mean |d ln y / d ln theta| over the conditions (rows: outputs, columns: free parameters)


,K_g6p_1,Ks_g6p_pgi,kcat_f_2,kcat_f_3,Ks_fbp_4,kcat_f_4,kcat_f_5,kcat_f_6,kcat_f_7,kcat_f_8,kcat_f_9
output,,,,,,,,,,,
C_3pg,1.686847e-03,2.740746e-03,3.887754e-03,0.329928,0.017056,0.308871,0.006185,0.003575,0.018106,0.008056,0.397823
C_dhap,8.017005e-10,1.944724e-04,3.019428e-04,0.044214,0.003787,0.146393,0.000141,0.000104,0.000276,0.000058,0.004932
C_f6p,1.622586e-04,3.709638e-03,6.632643e-03,0.042593,0.002565,0.018761,0.000218,0.000247,0.000386,0.000237,0.009383
C_fbp,2.129978e-07,6.790082e-04,1.306780e-03,0.218530,0.031085,0.155371,0.000141,0.000243,0.001041,0.000301,0.034769
C_g6p,1.430181e-04,7.102008e-03,1.284678e-02,0.038492,0.002836,0.017398,0.000193,0.000231,0.000346,0.000388,0.008221
C_pep,2.079569e-03,2.857783e-03,3.636546e-03,0.289387,0.015954,0.421351,0.006066,0.003816,0.017650,0.013041,0.314596
v_gapA,2.045928e-04,3.203932e-04,4.476571e-04,0.036543,0.005108,0.152527,0.000463,0.000884,0.000473,0.000376,0.016974
v_gpmA,2.022128e-04,3.259453e-04,4.615293e-04,0.039042,0.005414,0.162200,0.000466,0.000936,0.000458,0.000690,0.019368
v_pfkB,2.004237e-05,2.884626e-04,5.045840e-04,0.066862,0.017920,0.478428,0.000060,0.000309,0.000425,0.000824,0.016396


objective of the literature model (mean over conditions): 7771.70


## 2. Metrics per stage

The in-sample data of the final analysis (`final/pred_insample_real.csv`) is the common reference. Predictions: the literature inner problem, the stage-0 and final in-sample predictions, the LOOCV held-out predictions, and the LOOCV training-mean baseline. Every $R^2$ uses the per-output mean of the in-sample data as the reference value; `n` and the RMSE are over the cells that are measured in the data.

In [5]:
real_path = os.path.join(FINAL, "pred_insample_real.csv")
sources = {
    "literature": os.path.join(DATA_OUT, "y_literature_inner.csv"),
    "stage0": os.path.join(OUT, "stage0", "pred_insample_predicted.csv"),
    "final": os.path.join(FINAL, "pred_insample_predicted.csv"),
    "loocv": os.path.join(OUT, "loocv", "loocv_pred.csv"),
}
baseline_path = os.path.join(OUT, "loocv", "loocv_baseline.csv")
missing = [k for k, v in sources.items() if not os.path.exists(v)] + ([] if os.path.exists(baseline_path) else ["loocv baseline"])

if not os.path.exists(real_path) or missing:
    print("cannot build stage_metrics.csv yet; missing:", missing or ["final/pred_insample_real.csv"])
    stage_metrics = None
else:
    real = pd.read_csv(real_path, index_col=0)
    stages = {k: pd.read_csv(v, index_col=0).reindex(index=real.index, columns=real.columns) for k, v in sources.items()}
    baseline = pd.read_csv(baseline_path, index_col=0).reindex(index=real.index, columns=real.columns)
    reference = {c: float(real[c].mean()) for c in real.columns}

    table = []
    for name, pred in stages.items():
        m = pred_metrics(real, pred, met_ref=reference, flux_ref=reference)
        m["stage"] = name
        table.append(m)
    m = pred_metrics(real, baseline, met_ref=reference, flux_ref=reference)
    m["stage"] = "loocv_training_mean_baseline"
    table.append(m)
    stage_metrics = pd.DataFrame(table).set_index("stage")

    stage0_row = json.load(open(os.path.join(OUT, "stage0", "stage_row.json")))
    final_state = json.load(open(os.path.join(FINAL, "final_summary.json")))
    stage_metrics["obj"] = [float(bd_lit.loc["MEAN", "total"]) if bd_lit is not None else np.nan, stage0_row["obj"], final_state["obj"], np.nan, np.nan]
    if WRITABLE:
        os.makedirs(DATA_OUT, exist_ok=True)
        stage_metrics.to_csv(os.path.join(DATA_OUT, "stage_metrics.csv"))
        print("stage_metrics.csv written to", DATA_OUT)
    display(stage_metrics[["obj", "rmse_met", "r2_met_permean", "rmse_flux_li", "r2_flux_li_permean", "rmse_met_meanbaseline", "rmse_flux_li_meanbaseline"]])

stage_metrics.csv written to /Users/gabbi/Desktop/Repos/HybridKinetics-IIQ3733/results_v4/flux_model/paper_figs_data


,obj,rmse_met,r2_met_permean,rmse_flux_li,r2_flux_li_permean,rmse_met_meanbaseline,rmse_flux_li_meanbaseline
stage,,,,,,,
literature,7771.698706,0.228400,-1.101202,3.743320,-25.508696,0.157566,0.727047
stage0,11.745483,0.120675,0.413445,0.411894,0.679044,0.157566,0.727047
final,10.942883,0.115092,0.466460,0.359586,0.755387,0.157566,0.727047
loocv,NaN,0.203540,-0.668703,1.376363,-2.583773,0.157566,0.727047
loocv_training_mean_baseline,NaN,0.165854,-0.107972,0.761669,-0.097506,0.157566,0.727047
